# Using Tools with Claude

In [74]:
from dotenv import load_dotenv
import anthropic
from anthropic import Anthropic

print(f"Using Anthropic API: {anthropic.__version__}")

load_dotenv(override=True)

client = Anthropic()
MODEL = "claude-haiku-4-5"
MAX_TOKENS = 1024

Using Anthropic API: 1.1.0


### Define the tool functions

Let's define a tool function to get current date & time in a given format.

In [75]:
from datetime import datetime


def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be null")
    return datetime.now().strftime(date_format)

In [76]:
# some test calls
print(f"Default: {get_current_datetime()}")
print(f"Custom: {get_current_datetime('%d/%m/%Y %I:%M %p')}")

Default: 2026-09-08 11:31:49
Custom: 08/09/2026 11:31 AM


We will also need to create a JSON schema describing the tool call function & params. This can be generated using Claude AI. Following schema was generated by Claude AI, which we assign to a varible, named with the same name as the tool function and ending with `_schema`.


An easy way to create the schema is to ask Claude to generate it. Head over to [claude.ai](https://claude.ai) and type in the following prompt and paste the tool function below the prompt:

`"Write a valid JSON schema spec for the purposes of tool calling for this function. Follow the best practices listed in the attached documentation available at https://platform.claude.com/docs/en/agents-and-tools/tool-use/overview.md"`

📌 **NOTE**: as of Aug 2026, the URL for Anthropic's tool documentation is - `https://platform.claude.com/docs/en/agents-and-tools/tool-use/overview.md` - this could change in future. Paste in the correct URL.

The cell below shows me what I got back from Claude as the tool schema.

In [77]:
get_current_datetime_schema = {
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes.",
                "default": "%Y-%m-%d %H:%M:%S",
            }
        },
        "required": [],
    },
}

Now let's call Claude with this tool schema (JSON) and a user query. 

In [78]:
messages = []

messages.append(
    {"role": "user", "content": "What is the exact time formatted as HH:MM:SS?"}
)

response = client.messages.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    messages=messages,
    tools=[get_current_datetime_schema],
)
messages.append({"role": "assistant", "content": response.content})

print(response)

Message(id='msg_011Cequ2wbZL6QNJuAzV8MH7', container=None, content=[ToolUseBlock(id='toolu_013yZJUFgDxtaVYGw2kR2Ej2', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=618, output_tokens=63, output_tokens_details=None, server_tool_use=None, service_tier='standard'))


Here's how you'd run the tool itself. This code is written like this because Anthropic does not guarantee that every multi-block will always contain a text block.

In [79]:
from typing import NamedTuple, List, Dict
from anthropic.types import Message


class ToolCall(NamedTuple):
    id: str
    name: str
    input: dict


def get_tool_calls(response: Message) -> list[ToolCall]:
    """Return a ToolCall for every tool_use block in the response, or []."""
    if response.stop_reason != "tool_use":
        return []
    return [
        ToolCall(block.id, block.name, block.input)
        for block in response.content
        if block.type == "tool_use"
    ]


def make_tools_calls(response: Message) -> List[Dict]:
    """calls the respective tool and packs the result as expected by Claude"""

    tool_blocks = []
    if response.stop_reason == "tool_use":
        # tool_blocks = [block for block in response.content if block.type == "tool_use"]
        tool_blocks = get_tool_calls(response)

    # now call our tool - if tool_blocks is [], then this block will
    # not return anything!
    tool_call_results = []
    for tool_block in tool_blocks:
        name = tool_block.name  # which tool Claude wants to call
        args = tool_block.input  # dict of arguments for that tool
        tool_use_id = tool_block.id  # needed when you send the result back

        # check which tool function was asked for
        # in this example, there is just 1, but this could unwind to
        # a block like this...

        if name == "get_current_datetime":
            print(f"Calling tool {name} with args: {args}")
            result = get_current_datetime(**args)
            print(f"Result from {name}: {result}")
        # elsif name == "another_tool_name":
        #     result = another_tool_function(**args)

        # NOTE: only one of the tool function will be called at any time
        tool_call_result = {
            "type": "tool_result",
            "tool_use_id": tool_use_id,
            "content": result,
            "is_error": False,
        }
        tool_call_results.append(tool_call_result)

    return tool_call_results

In [80]:
# make the tool call
tool_call_results = make_tools_calls(response)
print(tool_call_results)

Calling tool get_current_datetime with args: {'date_format': '%H:%M:%S'}
Result from get_current_datetime: 11:31:50
[{'type': 'tool_result', 'tool_use_id': 'toolu_013yZJUFgDxtaVYGw2kR2Ej2', 'content': '11:31:50', 'is_error': False}]


Now we need to pass back the tool call result to Claude & get it's response. Here is how you do that.

In [81]:
messages.append({"role": "user", "content": tool_call_results})
print(messages)

[{'role': 'user', 'content': 'What is the exact time formatted as HH:MM:SS?'}, {'role': 'assistant', 'content': [ToolUseBlock(id='toolu_013yZJUFgDxtaVYGw2kR2Ej2', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)]}, {'role': 'user', 'content': [{'type': 'tool_result', 'tool_use_id': 'toolu_013yZJUFgDxtaVYGw2kR2Ej2', 'content': '11:31:50', 'is_error': False}]}]


In [82]:
final_response = client.messages.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    messages=messages,
    tools=[get_current_datetime_schema],
)

print(final_response.content[0].text)

The exact time is **11:31:50** (HH:MM:SS format).
